#1. Read Bronze

In [0]:
from pyspark.sql import functions as F

catalog = "adb_ecommerce_de_7405607562556202"

order_items_df = spark.table(
    f"{catalog}.ecommerce_bronze.order_items"
)

#2. Clean + standardize

For this dataset, we'll:
1. remove duplicate order-item records
2. trim IDs
3. ensure numeric fields have proper types
4. keep the source structure clean

In [0]:
silver_order_items = (
    order_items_df
    .dropDuplicates(["order_id", "order_item_id"])
    .withColumn("order_id", F.trim(F.col("order_id")))
    .withColumn("product_id", F.trim(F.col("product_id")))
    .withColumn("seller_id", F.trim(F.col("seller_id")))
    .withColumn("shipping_limit_date", F.col("shipping_limit_date").cast("timestamp"))
    .withColumn("price", F.col("price").cast("double"))
    .withColumn("freight_value", F.col("freight_value").cast("double"))
)

#3. Write to ADLS

In [0]:
silver_path = "abfss://ecommerce@mayankecommerce017.dfs.core.windows.net/silver/order_items"

silver_order_items.write \
    .format("delta") \
    .mode("overwrite") \
    .save(silver_path)

#4. Register in Unity Catalog

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.ecommerce_silver.order_items
USING DELTA
LOCATION '{silver_path}'
""")